In [1]:
!pip install -U python-jobspy playwright
!playwright install firefox

Defaulting to user installation because normal site-packages is not writeable
  Using cached python_jobspy-1.1.82-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 13.2 MB/s  0:00:01 13.5 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached markdownify-0.13.1-py3-none-any.whl.metadata (8.5 kB)
  Using cached pandas-2.3.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tls_client-1.0.1-py3-none-any.whl.metadata (5.0 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metad

In [4]:
import csv
import pandas as pd
from jobspy import scrape_jobs
from naukri_scraper import scrape_naukri

# 1. Scrape fast API-based job boards via JobSpy
jobs_jobspy = scrape_jobs(
    site_name=["indeed", "linkedin", "glassdoor", "google", "bayt"],
    search_term="python",
    google_search_term="python jobs in Mumbai since yesterday",
    location="Mumbai, Maharashtra",
    results_wanted=20,
    hours_old=130,
    country_indeed="India",
    linkedin_fetch_description=True
)

# 2. Scrape Naukri via local headless browser (bypasses Akamai/406)
jobs_naukri = scrape_naukri(
    search_term="python",
    location="Mumbai",
    results_wanted=20,
    hours_old=130
)

# 3. Combine both into unified dataset
jobs = pd.concat([jobs_jobspy, jobs_naukri], ignore_index=True)
print(f"Total jobs collected: {len(jobs)}")
print(jobs["site"].value_counts())


2026-09-05 14:28:42,042 - ERROR - JobSpy:Glassdoor - Glassdoor response status code 400
2026-09-05 14:28:42,043 - ERROR - JobSpy:Glassdoor - Glassdoor: location not parsed


In [6]:
jobs.head()

import os, datetime

folder = "scrape_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(folder, exist_ok=True)
jobs.to_csv(os.path.join(folder, "jobs.csv"), quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False) # to_excel
print("saved to:", folder)


40